# Bayesian Optimization for kinematic-fit parameters

Recover $(x_0, v_0, a)$ of $x(t) = x_0 + v_0 t + \tfrac{1}{2} a t^2$ from synthetic noisy observations by maximizing $-\text{MSE}$ with a `SingleTaskGP` + `qLogExpectedImprovement`.

## Imports and constants

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.utils.sampling import draw_sobol_samples
from gpytorch.mlls import ExactMarginalLogLikelihood

%matplotlib inline

In [ ]:
DTYPE = torch.double
DEVICE = torch.device("cpu")

TRUE_PARAMS = (1.0, 2.0, -9.81)
NOISE_STD = 0.05
N_INIT = 12
N_ITER = 40
Q = 1

# Fix 2: tighter BOUNDS. Prior knowledge "falling object near Earth gravity"
# narrows the search box from the original [-5,5]^2 x [-20,20] to a region
# better matched to plausible (x0, v0, a). Same evaluation budget covers the
# relevant region with much higher sample density.
BOUNDS = torch.tensor(
    [[-3.0, -3.0, -15.0], [3.0, 3.0, -5.0]], dtype=DTYPE, device=DEVICE
)

torch.manual_seed(0)
np.random.seed(0)

## Synthetic data and forward model

`model_x` is vectorized over a batch of parameter sets `(q, 3)` and times `(T,)` to produce trajectories `(q, T)`. `objective` returns $-\text{MSE}$ with the trailing outcome dim BoTorch expects.

In [ ]:
def make_data(true_params, t, noise_std):
    x0, v0, a = true_params
    x_clean = x0 + v0 * t + 0.5 * a * t**2
    x_obs = x_clean + noise_std * torch.randn_like(x_clean)
    return x_obs


def model_x(params: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    x0 = params[..., 0:1]
    v0 = params[..., 1:2]
    a = params[..., 2:3]
    t_row = t.unsqueeze(0)
    return x0 + v0 * t_row + 0.5 * a * t_row**2


def objective(params: torch.Tensor, t: torch.Tensor, x_obs: torch.Tensor) -> torch.Tensor:
    x_pred = model_x(params, t)
    mse = ((x_pred - x_obs.unsqueeze(0)) ** 2).mean(dim=-1, keepdim=True)
    return -mse

In [ ]:
t = torch.linspace(0.0, 2.0, 20, dtype=DTYPE, device=DEVICE)
x_obs = make_data(TRUE_PARAMS, t, NOISE_STD)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(t.numpy(), x_obs.numpy(), color="k", s=20, label="observed")
ax.set_xlabel("t"); ax.set_ylabel("x(t)"); ax.legend(); ax.set_title("Synthetic data")
plt.show()

## BO loop

Each iteration: refit a `SingleTaskGP` (with `Normalize` on inputs and `Standardize` on outcomes), build `qLogEI`, optimize the acquisition with multi-start L-BFGS-B, evaluate the new candidate, and append it to the dataset.

In [ ]:
def run_bo(t: torch.Tensor, x_obs: torch.Tensor, verbose: bool = True):
    train_X = draw_sobol_samples(bounds=BOUNDS, n=N_INIT, q=1, seed=0).squeeze(1).to(
        dtype=DTYPE, device=DEVICE
    )
    train_Y = objective(train_X, t, x_obs)

    history = []

    for it in range(N_ITER):
        model = SingleTaskGP(
            train_X,
            train_Y,
            input_transform=Normalize(d=BOUNDS.shape[-1], bounds=BOUNDS),
            outcome_transform=Standardize(m=1),
        )
        mll = ExactMarginalLogLikelihood(model.likelihood, model)
        fit_gpytorch_mll(mll)

        acq = qLogExpectedImprovement(model=model, best_f=train_Y.max())
        candidate, _ = optimize_acqf(
            acq_function=acq,
            bounds=BOUNDS,
            q=Q,
            num_restarts=10,
            raw_samples=256,
        )

        new_y = objective(candidate, t, x_obs)
        train_X = torch.cat([train_X, candidate], dim=0)
        train_Y = torch.cat([train_Y, new_y], dim=0)

        best_idx = int(torch.argmax(train_Y).item())
        best_neg_mse = float(train_Y[best_idx].item())
        best_mse = -best_neg_mse
        bx0, bv0, ba = [float(v) for v in train_X[best_idx].tolist()]
        history.append(best_mse)
        if verbose:
            print(
                f"iter {it+1:3d}/{N_ITER} | best -MSE = {best_neg_mse: .6e} "
                f"(MSE = {best_mse:.6e}) | x0={bx0: .4f} v0={bv0: .4f} a={ba: .4f}"
            )

    return train_X, train_Y, history

In [ ]:
train_X, train_Y, history = run_bo(t, x_obs)

## Results

In [ ]:
best_idx = int(torch.argmax(train_Y).item())
best_params = train_X[best_idx]
best_mse = float(-train_Y[best_idx].item())
bx0, bv0, ba = [float(v) for v in best_params.tolist()]
tx0, tv0, ta = TRUE_PARAMS

print("=" * 60)
print(f"Best params: x0={bx0:.6f} v0={bv0:.6f} a={ba:.6f}")
print(f"True params: x0={tx0:.6f} v0={tv0:.6f} a={ta:.6f}")
print(f"Errors:      dx0={bx0-tx0:+.6f} dv0={bv0-tv0:+.6f} da={ba-ta:+.6f}")
print(f"Best MSE:    {best_mse:.6e} (noise variance ~ {NOISE_STD**2:.6e})")
print("=" * 60)

### Fix 1: local polish with L-BFGS-B

BO finds the right *basin*; L-BFGS-B finds the *bottom* of that basin. The objective is differentiable in `(x0, v0, a)`, so a few dozen gradient-based steps from BO's best point typically drop MSE to the noise floor.

In [ ]:
from scipy.optimize import minimize

bo_best_params = best_params.clone()
bo_best_mse = best_mse

def neg_obj_np(p):
    pt = torch.tensor(p, dtype=DTYPE, device=DEVICE).unsqueeze(0)
    return -float(objective(pt, t, x_obs).item())

scipy_bounds = list(zip(BOUNDS[0].tolist(), BOUNDS[1].tolist()))
res = minimize(
    neg_obj_np,
    x0=bo_best_params.cpu().numpy(),
    method="L-BFGS-B",
    bounds=scipy_bounds,
)

polished_params = torch.tensor(res.x, dtype=DTYPE, device=DEVICE)
polished_mse = -float(objective(polished_params.unsqueeze(0), t, x_obs).item())
px0, pv0, pa = [float(v) for v in polished_params.tolist()]

print("=" * 60)
print(f"BO best MSE:        {bo_best_mse:.6e}")
print(f"Polished MSE:       {polished_mse:.6e}")
print(f"Noise variance:     {NOISE_STD**2:.6e}  (irreducible floor)")
print("-" * 60)
print(f"BO best params:     x0={bx0: .6f} v0={bv0: .6f} a={ba: .6f}")
print(f"Polished params:    x0={px0: .6f} v0={pv0: .6f} a={pa: .6f}")
print(f"True params:        x0={tx0: .6f} v0={tv0: .6f} a={ta: .6f}")
print(f"Polished errors:    dx0={px0-tx0:+.6f} dv0={pv0-tv0:+.6f} da={pa-ta:+.6f}")
print(f"L-BFGS-B converged: {res.success}  ({res.nit} iters, {res.nfev} fevals)")
print("=" * 60)

# Use polished params for downstream best-fit plot.
best_params = polished_params
best_mse = polished_mse

### Best-fit trajectory

In [ ]:
t_dense = torch.linspace(float(t.min()), float(t.max()), 200, dtype=DTYPE, device=DEVICE)
x_true = TRUE_PARAMS[0] + TRUE_PARAMS[1] * t_dense + 0.5 * TRUE_PARAMS[2] * t_dense**2
x_fit = model_x(best_params.unsqueeze(0), t_dense).squeeze(0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(t.numpy(), x_obs.numpy(), color="k", s=20, label="observed")
ax.plot(t_dense.numpy(), x_true.numpy(), "g--", label="true")
ax.plot(t_dense.numpy(), x_fit.numpy(), "r-", label="best fit")
ax.set_xlabel("t"); ax.set_ylabel("x(t)"); ax.legend()
ax.set_title("Kinematic fit via BoTorch BO")
plt.show()

### Convergence

In [ ]:
per_eval_mse = (-train_Y.squeeze(-1)).cpu().numpy()
eval_idx = np.arange(1, len(per_eval_mse) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(range(1, len(history) + 1), history, "b-o", markersize=3)
ax.axhline(NOISE_STD**2, color="gray", linestyle=":", label=f"noise variance ({NOISE_STD**2:.2e})")
ax.set_yscale("log")
ax.set_xlabel("BO iteration"); ax.set_ylabel("best MSE so far")
ax.set_title("BO convergence (running min)"); ax.grid(True, which="both", alpha=0.3); ax.legend()

ax = axes[1]
ax.plot(eval_idx[:N_INIT], per_eval_mse[:N_INIT], "o", color="C1", markersize=4, label="Sobol init")
ax.plot(eval_idx[N_INIT:], per_eval_mse[N_INIT:], "o-", color="C0", markersize=4, label="BO proposals")
ax.axvline(N_INIT + 0.5, color="gray", linestyle="--", alpha=0.5)
ax.axhline(NOISE_STD**2, color="gray", linestyle=":", label=f"noise variance ({NOISE_STD**2:.2e})")
ax.set_yscale("log")
ax.set_xlabel("evaluation #"); ax.set_ylabel("MSE at this evaluation")
ax.set_title("Per-evaluation loss"); ax.grid(True, which="both", alpha=0.3); ax.legend()

fig.tight_layout()
plt.show()